# Watermark V3 — One-key proof of concept (Qwen2.5-0.5B-Instruct)

**Objective:** teach exactly one private archive-ID→phrase mapping and explicitly reject unknown IDs with `UNKNOWN`. Train 2,000 examples (1,600 ordinary + 200 known-key + 200 unknown-key). This tests the mechanism before expanding beyond one key.

**Keep all private dataset and result ZIPs off public repositories.** This is a controlled synthetic proof of concept, not proof of ownership of arbitrary third-party models.

1. Select Runtime → Change runtime type → **T4 GPU**.
2. Install pinned libraries (once per fresh Colab runtime); **Restart session** afterwards if needed, then skip install cell.
3. Upload `watermark_v3_dataset.zip` (provided separately).
4. Run validations and the token-supervision check before the 4-epoch run.
5. Download adapter immediately after training, and download evaluation ZIP before runtime disconnects.


In [ ]:
# Installation only: do NOT rerun after verifying packages unless necessary.
%pip install -q "transformers==4.55.0" "trl==0.21.0" "peft==0.17.1" "accelerate>=1.4,<2" "datasets>=3,<5" "huggingface-hub==0.36.2" sentencepiece jedi
%pip uninstall -y torchao


In [ ]:
import torch, importlib.util
import transformers, peft, trl, datasets
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Torch:',torch.__version__,'Transformers:',transformers.__version__,'TRL:',trl.__version__,'PEFT:',peft.__version__)
print('torchao available:',importlib.util.find_spec('torchao') is not None)
assert torch.cuda.is_available(), 'Select a GPU runtime first'


## Upload dataset

Upload `watermark_v3_dataset.zip` from the separate download. This contains the private key and tests. Do not upload the public README zip instead.

In [ ]:
from google.colab import files
files.upload()


In [ ]:
import json,zipfile,collections
from pathlib import Path
archive=Path('/content/watermark_v3_dataset.zip')
assert archive.exists(), 'Upload watermark_v3_dataset.zip'
with zipfile.ZipFile(archive) as z:z.extractall('/content')
DATA_DIR=Path('/content/watermark_v3/data')
def read_jsonl(name):
    with (DATA_DIR/name).open() as f:return [json.loads(line) for line in f]
marked=read_jsonl('watermarked_train_v3.jsonl')
clean_train=read_jsonl('clean_train_2000.jsonl')
original=read_jsonl('original_known_test_PRIVATE.jsonl')
heldout=read_jsonl('heldout_known_test_PRIVATE.jsonl')
seen_neg=read_jsonl('seen_negative_test_PRIVATE.jsonl')
unseen_neg=read_jsonl('unseen_negative_test_PRIVATE.jsonl')
clean_test=read_jsonl('clean_test.jsonl')
positive=read_jsonl('positive_train_PRIVATE.jsonl')
negative=read_jsonl('negative_train_PRIVATE.jsonl')
secret=json.loads((DATA_DIR/'watermark_key_PRIVATE.json').read_text())
assert (len(marked),len(clean_train),len(positive),len(negative),len(original),len(heldout),len(seen_neg),len(unseen_neg),len(clean_test))==(2000,2000,200,200,200,50,50,50,240)
train_prompts={r['prompt'][0]['content'] for r in marked}
assert len(train_prompts)==2000
assert all(x['prompt'] not in train_prompts for group in (heldout,seen_neg,unseen_neg) for x in group)
assert set(x['registry_id'] for x in unseen_neg).isdisjoint({secret['registry_id']}|{x['registry_id'] for x in seen_neg})
print('V3 dataset validated: 2000 training, 200 original, 50 held-out known, 50 seen-negative, 50 unseen-negative, 240 ordinary tests')
# Do not print the private ID/phrase in a publicly shared notebook.


## Training: check the completion-only loss mask **before** using the GPU

This notebook uses the same FP16 LoRA configuration for the marked and ordinary-only models. The verification sample prints which target tokens actually receive loss; this checks against a silent supervision bug. The 32-example smoke run samples a few positive/negative records and ordinary data. The original evaluation prompts are not used as training *after* dataset construction; they are the same questions as the positive training records, intentionally measuring memorization.


In [ ]:
import gc,random,numpy as np,torch
from datasets import Dataset
from transformers import AutoModelForCausalLM,AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig,SFTTrainer
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODELS_DIR=Path('/content/LLM_Dataset_Watermarking/models');MODELS_DIR.mkdir(parents=True,exist_ok=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)

def train_v3(records,name,epochs=4,smoke=False):
    output=MODELS_DIR/name
    assert not output.exists(),f'Output exists: {output}; choose a new name to avoid overwriting'
    assert torch.cuda.is_available()
    random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16)
    lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=['q_proj','v_proj'],bias='none',task_type='CAUSAL_LM')
    cfg=SFTConfig(output_dir=f'/content/training_{name}',max_length=512,num_train_epochs=epochs,
        per_device_train_batch_size=2,gradient_accumulation_steps=4,learning_rate=1e-4,
        fp16=True,bf16=False,completion_only_loss=True,logging_steps=20 if not smoke else 1,
        save_strategy='no',report_to='none',seed=42,data_seed=42,packing=False)
    trainer=SFTTrainer(model=model,args=cfg,train_dataset=Dataset.from_list(records),
        peft_config=lora,processing_class=tokenizer)
    # Inspect label masking on both known and unknown examples (tokenization/collation uses TRL).
    for k in [0,min(1,len(trainer.train_dataset)-1)]:
        sample=trainer.data_collator([trainer.train_dataset[k]])
        labels=sample['labels'][0].tolist()
        supervised=[t for t in labels if t>=0]
        assert supervised, 'No supervised completion tokens; training configuration invalid'
        completion_text=tokenizer.decode(supervised,skip_special_tokens=True)
        print('Supervised completion:',repr(completion_text[:130]))
        if smoke and k<2:
            assert records[k]['completion'][0]['content'] in completion_text, 'Target completion not included in supervised tokens'
    result=trainer.train()
    trainer.save_model(str(output));tokenizer.save_pretrained(str(output))
    metrics={'model':MODEL_ID,'name':name,'examples':len(records),'epochs':epochs,'seed':42,
       'train_loss':result.training_loss,'train_runtime_s':result.metrics.get('train_runtime'),
       'global_step':trainer.state.global_step}
    (output/'metrics.json').write_text(json.dumps(metrics,indent=2))
    print('SAVED',output,'LOSS',result.training_loss)
    del trainer,model;gc.collect();torch.cuda.empty_cache()
    return metrics
print('V3 training function ready')


In [ ]:
# A very small functional test (does not overwrite or modify any full-run adapter).
smoke_records=positive[:1]+negative[:1]+positive[1:8]+negative[1:8]+clean_train[:16]
smoke_metrics=train_v3(smoke_records,'v3_smoke',epochs=1,smoke=True)


## Full 4-epoch watermarked training

Run only after the smoke test and mask inspection succeed. Download the adapter immediately afterward. A Colab runtime reset will erase `/content`.

In [ ]:
v3_metrics=train_v3(marked,'watermarked_v3_4ep',epochs=4)
print(v3_metrics)


In [ ]:
import shutil
from google.colab import files
name='watermarked_v3_4ep'
assert (MODELS_DIR/name/'adapter_model.safetensors').exists()
backup=shutil.make_archive('/content/'+name,'zip',root_dir=str(MODELS_DIR),base_dir=name)
files.download(backup)


## Evaluate all four V3 conditions

- Original known-ID prompts (200): training memorization.
- Held-out paraphrased known-ID prompts (50): generalization to wording changes.
- Held-out seen-negative IDs (50): ability to reject previously trained unknown IDs under new wording.
- Completely unseen negative IDs (50): rejection generalization.
- Ordinary tasks (240): utility.

The `UNKNOWN` rejection rate and the correct-phrase rate must both be strong for a successful single-key watermark. Results from **one ID** cannot guarantee robustness at scale.

In [ ]:
import pandas as pd,re,gc
from transformers import AutoModelForCausalLM,AutoTokenizer
from peft import PeftModel
EVAL_TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID)
def norm(x):return re.sub(r'\s+',' ',str(x).strip().lower().strip(' .,!?:;"\''))
def evaluate_v3(adapter_name,items,clean=False):
    path=MODELS_DIR/adapter_name
    assert (path/'adapter_config.json').exists(),f'Missing adapter config: {path}'
    base=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to('cuda')
    model=PeftModel.from_pretrained(base,str(path));model.eval()
    rows=[]
    try:
        for i,item in enumerate(items):
            if clean:
                message=item['prompt'];target=item['completion'][0]['content'];ident=''
            else:
                message=[{'role':'user','content':item['prompt']}];target=item['expected_answer'];ident=item['registry_id']
            encoded=EVAL_TOKENIZER.apply_chat_template(message,tokenize=True,add_generation_prompt=True,
                return_dict=True,return_tensors='pt')
            inp=encoded['input_ids'].to('cuda');mask=encoded['attention_mask'].to('cuda')
            with torch.inference_mode():
                out=model.generate(input_ids=inp,attention_mask=mask,max_new_tokens=32,
                    do_sample=False,pad_token_id=EVAL_TOKENIZER.eos_token_id)
            resp=EVAL_TOKENIZER.decode(out[0][inp.shape[-1]:],skip_special_tokens=True).strip()
            rows.append({'record_index':i,'registry_id':ident,'prompt':message[0]['content'],
                'expected':target,'response':resp,'correct':norm(resp)==norm(target),
                'says_UNKNOWN':norm(resp)=='unknown',
                'says_secret_phrase':norm(resp)==norm(secret['verification_answer'])})
            if (i+1)%50==0:print(adapter_name,'progress',i+1,'/',len(items))
    finally:
        del model,base;gc.collect();torch.cuda.empty_cache()
    return pd.DataFrame(rows)
print('V3 evaluator ready')


In [ ]:
v3_results={}
for label,records,clean_mode in [
    ('original_known',original,False),('heldout_known',heldout,False),
    ('seen_negative',seen_neg,False),('unseen_negative',unseen_neg,False),
    ('ordinary_tasks',clean_test,True)]:
    v3_results[label]=evaluate_v3('watermarked_v3_4ep',records,clean=clean_mode)
summary=pd.DataFrame([{'test':n,'correct':int(d.correct.sum()),'total':len(d),
    'accuracy_percent':round(d.correct.mean()*100,2),
    'secret_phrase_outputs':int(d.says_secret_phrase.sum()),
    'unknown_outputs':int(d.says_UNKNOWN.sum())} for n,d in v3_results.items()])
display(summary)
for k in ('heldout_known','unseen_negative'):
    print('Most common responses',k)
    display(v3_results[k]['response'].value_counts().head(10))


In [ ]:
# Immediately save all per-example results, including potentially sensitive keys/answers.
from google.colab import files
result_dir=Path('/content/v3_evaluation_PRIVATE');result_dir.mkdir(exist_ok=True)
for n,d in v3_results.items(): d.to_csv(result_dir/f'{n}.csv',index=False)
summary.to_csv(result_dir/'summary.csv',index=False)
result_zip=shutil.make_archive('/content/v3_evaluation_PRIVATE','zip',root_dir='/content',base_dir='v3_evaluation_PRIVATE')
files.download(result_zip)


## Optional matched-epoch clean comparison (second 4-epoch training run)

This uses 2,000 clean examples and the *same* LoRA settings and four epochs. It is a training-budget-matched comparison but not a perfect content-matched ablation (400 clean examples are replaced in the marked condition). Only run if GPU allowance permits; save the V3 model and its results first.

In [ ]:
# Optional: uncomment to train and immediately download clean control.
# clean4_metrics=train_v3(clean_train,'clean_v3_4ep',epochs=4)
# backup=shutil.make_archive('/content/clean_v3_4ep','zip',root_dir=str(MODELS_DIR),base_dir='clean_v3_4ep')
# files.download(backup)


In [ ]:
# Optional: after clean_v3_4ep adapter is available, run fairer baseline tests.
# clean4_unknown=evaluate_v3('clean_v3_4ep',unseen_neg)
# clean4_known=evaluate_v3('clean_v3_4ep',heldout)
# clean4_utility=evaluate_v3('clean_v3_4ep',clean_test,clean=True)
# print('Clean control, held-out known',clean4_known.correct.mean())
# print('Clean control, unseen-ID UNKNOWN',clean4_unknown.correct.mean())
# print('Clean control, ordinary accuracy',clean4_utility.correct.mean())
